In [43]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import FinanceDataReader as fdr

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    confusion_matrix
)
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# =========================================================
# 기본 설정
# =========================================================

### 데이터 티커 및 기간 설정
ETF_CODE = "SMH"
START_DATE = "2020-01-01"
END_DATE = None

### Target 설정
N_DAYS = 5                                  # N일 후의 상승/하락 예측
THRESHOLD = 0.01                            # 상승/하락 판단 기준 (예: 0.05는 5% 상승/하락)
VALID_MONTHS = 1                            # 검증 데이터 기간 (개월 단위, 예: 1은 최근 1개월)


### 변수 선택시 
VIF_THRESHOLD = 10                          # 변수 선택시 VIF 기준 (예: 10 이상인 변수 제거)
LAG_SEARCH_YEARS = 1                        # 변수별 최적 lag 탐색시 사용할 최근 데이터 기간 (년 단위)
LAG_DAYS = [1, 3, 5, 10, 20, 40, 60, 120]   # 변수별 최적 lag 탐색시 사용할 일수 (예: 1, 3, 5, 10, 20, 40, 60, 120일)

### RF 설정 (변수 중요도 파악을 위한 과적합용 모델)
RANDOM_STATE = 42
N_RF_RUNS = 3
N_REPEATS = 10
TOP_N = 30 # 상위 N개 변수 선택
PRED_THRESHOLD = 0.5


# 외부 지표
EXTERNAL_TICKERS = {
    "QQQ": "QQQ",
    "SPY": "SPY",
    "SOXX": "SOXX",
    "NVDA": "NVDA",
    "TSM": "TSM",
    "VIX": "^VIX",
    "TNX": "^TNX",
    "USDKRW": "KRW=X",
    "DXY": "DX-Y.NYB",
    "GOLD": "GC=F",
    "OIL": "CL=F",
}

EXTERNAL_FEATURE_TYPES = {
    "QQQ": "price",
    "SPY": "price",
    "SOXX": "price",
    "NVDA": "price",
    "TSM": "price",
    "VIX": "risk",
    "TNX": "rate",
    "USDKRW": "price",
    "DXY": "price",
    "GOLD": "price",
    "OIL": "price",
}



In [44]:
etf_list = fdr.StockListing("ETF/KR")
etf_list.head()

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,117200,5,-7655,-6.13,116911.0,43.8068,27724693,3333867,249988
1,360750,4,TIGER 미국S&P500,27765,2,170,0.62,27919.0,13.8882,19449232,540600,179445
2,396500,2,TIGER 반도체TOP10,46300,5,-3765,-7.52,45966.0,58.1478,28590462,1371155,128575
3,133690,4,TIGER 미국나스닥100,194300,5,-620,-0.32,196648.0,23.3675,1241734,242445,102746
4,102110,1,TIGER 200,117285,5,-7485,-6.00,116956.0,43.9619,8171287,980993,101862


In [45]:
## 함수

################################################################################################################################################
## 1. Base Feature Dataset 생성
################################################################################################################################################
def load_price_data(ticker, start_date="2020-01-01", end_date=None):
    """
    FinanceDataReader로 가격 데이터를 가져온다.
    Date 컬럼을 일반 컬럼으로 유지한다.
    """
    df = fdr.DataReader(ticker, start_date, end_date)
    df = df.reset_index().rename(columns={"index": "Date"})

    # 컬럼명 정리
    df["Date"] = pd.to_datetime(df["Date"])

    return df

def make_target_etf_features(etf_df, prefix):
    """
    예측 대상 ETF용 feature 생성.
    target은 여기서 만들지 않는다.
    """
    df = etf_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]
    volume = df["Volume"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    # target 만들 때 필요하므로 Close는 반드시 보존
    result[f"{prefix}_adj_close"] = close

    # 수익률
    result[f"{prefix}_ret_1d"] = close.pct_change(1)
    result[f"{prefix}_ret_5d"] = close.pct_change(5)
    result[f"{prefix}_ret_20d"] = close.pct_change(20)

    # 이동평균 대비 위치
    ma_5 = close.rolling(5).mean()
    ma_20 = close.rolling(20).mean()
    ma_60 = close.rolling(60).mean()

    result[f"{prefix}_ma5_ratio"] = close / ma_5 - 1
    result[f"{prefix}_ma20_ratio"] = close / ma_20 - 1
    result[f"{prefix}_ma60_ratio"] = close / ma_60 - 1

    # 변동성
    result[f"{prefix}_vol_20d"] = result[f"{prefix}_ret_1d"].rolling(20).std()

    # 거래량 비율
    vol_ma20 = volume.rolling(20).mean()
    result[f"{prefix}_volume_ratio_20d"] = volume / vol_ma20 - 1

    return result

def make_external_features(raw_df, name, feature_type="price"):
    """
    외부 지표용 최소 파생변수 생성.
    feature_type:
        - price: 일반 가격형 지표
        - risk: VIX 같은 리스크 레벨 지표
        - rate: 금리 지표
    """
    df = raw_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    if feature_type == "price":
        result[f"{name}_ret_5d"] = close.pct_change(5)
        result[f"{name}_ret_20d"] = close.pct_change(20)

    elif feature_type == "risk":
        result[f"{name}_level"] = close
        result[f"{name}_chg_5d"] = close.diff(5)
        result[f"{name}_chg_20d"] = close.diff(20)

    elif feature_type == "rate":
        result[f"{name}_level"] = close
        result[f"{name}_diff_5d"] = close.diff(5)
        result[f"{name}_diff_20d"] = close.diff(20)

    else:
        raise ValueError("feature_type must be one of ['price', 'risk', 'rate']")

    return result

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.
    """
    # 1. ETF 본체 로드
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 2. ETF 본체 feature 생성
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # 3. 외부 지표 붙이기
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # 4. 날짜 정렬
    base_df = base_df.sort_values("Date").reset_index(drop=True)

    # 5. feature_cols 정리
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.

    주말 데이터가 섞여 NA가 늘어나는 문제를 막기 위해
    ETF / 외부 ticker 모두 영업일(월~금)만 사용한다.
    """

    # =========================================================
    # 0. 영업일 필터 함수
    # =========================================================
    def keep_weekdays_only(df, date_col="Date"):
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])
        df = df[df[date_col].dt.weekday < 5]  # 월=0, 금=4
        df = df.sort_values(date_col).reset_index(drop=True)
        return df

    # =========================================================
    # 1. ETF 본체 로드
    # =========================================================
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 주말 제거
    etf_raw = keep_weekdays_only(etf_raw, date_col="Date")

    # =========================================================
    # 2. ETF 본체 feature 생성
    # =========================================================
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # feature 생성 후에도 혹시 모르니 다시 주말 제거
    base_df = keep_weekdays_only(base_df, date_col="Date")

    # =========================================================
    # 3. 외부 지표 붙이기
    # =========================================================
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            # 외부 ticker도 주말 제거
            raw = keep_weekdays_only(raw, date_col="Date")

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            # 외부 feature 생성 후에도 다시 주말 제거
            ext_feat = keep_weekdays_only(ext_feat, date_col="Date")

            # ETF 거래일 기준으로 붙임
            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # =========================================================
    # 4. 날짜 정렬 + 주말 최종 제거
    # =========================================================
    base_df = keep_weekdays_only(base_df, date_col="Date")
    
    # =========================================================
    # 4.1. 외부 지표 NA 보정
    # - ETF 본체는 건드리지 않고
    # - 외부 지표만 ffill
    # =========================================================
    etf_prefix = f"{etf_code}_"

    external_cols = [
        col for col in base_df.columns
        if col != "Date" and not col.startswith(etf_prefix)
    ]

    base_df[external_cols] = base_df[external_cols].ffill()

    # =========================================================
    # 5. feature_cols 정리
    # =========================================================
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

################################################################################################################################################
## 2. VIF 기반 불필요 칼럼 제거
################################################################################################################################################
def reduce_features_by_vif(
    df,
    feature_cols,
    vif_threshold=30.0,
    date_col="Date",
    verbose=True
):
    """
    VIF 기준으로 다중공선성이 높은 feature를 반복 제거한다.

    주의:
    - target 생성 전 단계에서 실행한다.
    - lag 생성 전 단계에서 실행한다.
    - Date, adj_close 등 보존 컬럼은 feature_cols에 넣지 않는 것을 전제로 한다.
    """

    # 1. 숫자형 feature만 사용
    numeric_feature_cols = [
        col for col in feature_cols
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    work_df = df[numeric_feature_cols].copy()

    # 2. inf 처리
    work_df = work_df.replace([np.inf, -np.inf], np.nan)

    # 3. VIF 계산용 결측 제거
    #    여기서는 VIF 계산에만 dropna를 쓰고,
    #    원본 df 자체를 줄이지는 않는다.
    vif_calc_df = work_df.dropna(axis=0).copy()

    print("VIF 계산 대상 row 수:", len(vif_calc_df))
    print("VIF 계산 대상 feature 수:", len(numeric_feature_cols))

    if len(vif_calc_df) == 0:
        raise ValueError("VIF 계산 가능한 데이터가 없습니다. 결측값을 확인하세요.")

    # 4. 상수 컬럼 제거
    nunique = vif_calc_df.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()

    if len(constant_cols) > 0:
        print("상수 컬럼 제거:", constant_cols)

    remaining_cols = [
        col for col in numeric_feature_cols
        if col not in constant_cols
    ]

    removed_records = []

    # 5. VIF 반복 제거
    while True:
        if len(remaining_cols) <= 1:
            break

        X = vif_calc_df[remaining_cols].copy()

        # 표준화
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        vif_values = []

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_scaled, i)
            except Exception:
                vif = np.inf

            vif_values.append({
                "feature": col,
                "vif": vif
            })

        vif_df = pd.DataFrame(vif_values).sort_values("vif", ascending=False)

        max_vif_row = vif_df.iloc[0]
        max_feature = max_vif_row["feature"]
        max_vif = max_vif_row["vif"]

        if verbose:
            print(f"현재 max VIF: {max_vif:.2f} / feature: {max_feature}")

        if max_vif <= vif_threshold:
            break

        # 가장 VIF 높은 컬럼 제거
        remaining_cols.remove(max_feature)

        removed_records.append({
            "removed_feature": max_feature,
            "vif": max_vif,
            "remaining_feature_count": len(remaining_cols)
        })

    removed_vif_df = pd.DataFrame(removed_records)

    # 6. 최종 VIF 테이블 계산
    final_vif_records = []

    if len(remaining_cols) > 1:
        X_final = vif_calc_df[remaining_cols].copy()

        scaler = StandardScaler()
        X_final_scaled = scaler.fit_transform(X_final)

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_final_scaled, i)
            except Exception:
                vif = np.inf

            final_vif_records.append({
                "feature": col,
                "vif": vif
            })

        final_vif_df = pd.DataFrame(final_vif_records).sort_values("vif", ascending=False)

    else:
        final_vif_df = pd.DataFrame({
            "feature": remaining_cols,
            "vif": [np.nan] * len(remaining_cols)
        })

    print()
    print("========== VIF 제거 결과 ==========")
    print("초기 feature 수:", len(numeric_feature_cols))
    print("상수 제거 feature 수:", len(constant_cols))
    print("VIF 제거 feature 수:", len(removed_records))
    print("최종 feature 수:", len(remaining_cols))
    print("===================================")

    return remaining_cols, removed_vif_df, final_vif_df


################################################################################################################################################
## 3. Target 변수 생성
################################################################################################################################################
def add_target_column(
    df,
    close_col,
    n_days=5,
    threshold=0.05,
    target_col=None
):
    """
    현재 시점 기준 n_days 뒤 수익률이 threshold 이상이면 1, 아니면 0인 target 생성.

    예:
    n_days=5, threshold=0.05
    → 5거래일 뒤 수익률이 +5% 이상이면 target=1
    """

    result = df.copy()
    result = result.sort_values("Date").reset_index(drop=True)

    if target_col is None:
        target_col = f"target_{n_days}d_up_{int(threshold * 100)}pct"

    # 미래 가격
    future_close = result[close_col].shift(-n_days)

    # 미래 수익률
    result[f"future_ret_{n_days}d"] = future_close / result[close_col] - 1

    # target 생성
    result[target_col] = np.where(
        result[f"future_ret_{n_days}d"] >= threshold,
        1,
        0
    )

    # 마지막 n_days개는 미래 가격이 없으므로 제거
    result.loc[result[f"future_ret_{n_days}d"].isna(), target_col] = np.nan

    return result, target_col


################################################################################################################################################
## 4. Lag 생성 후 변수별 최적 lag 탐색
################################################################################################################################################

def find_best_lag_by_feature(
    df,
    feature_cols,
    target_col,
    lag_days=[1, 3, 5, 10, 20],
    date_col="Date",
    method="corr"
):
    """
    각 feature별로 target과 가장 관계가 강한 선행 lag를 찾는다.

    lag 의미:
    - lag=1  : feature의 1거래일 전 값으로 오늘 target 설명
    - lag=5  : feature의 5거래일 전 값으로 오늘 target 설명
    - lag=20 : feature의 20거래일 전 값으로 오늘 target 설명

    즉, feature가 먼저 움직이고 나중에 target이 움직이는 구조만 본다.
    """

    records = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        for lag in lag_days:
            temp = df[[date_col, col, target_col]].copy()

            # 선행변수 구조
            temp[f"{col}_lag{lag}"] = temp[col].shift(lag)

            temp = temp[[f"{col}_lag{lag}", target_col]].replace(
                [np.inf, -np.inf],
                np.nan
            ).dropna()

            if len(temp) < 30:
                continue

            x = temp[f"{col}_lag{lag}"]
            y = temp[target_col]

            if x.nunique() <= 1:
                corr = np.nan
            else:
                corr = x.corr(y)

            records.append({
                "feature": col,
                "lag": lag,
                "corr": corr,
                "abs_corr": abs(corr) if pd.notna(corr) else np.nan,
                "n_rows": len(temp)
            })

    lag_result_df = pd.DataFrame(records)

    if lag_result_df.empty:
        raise ValueError("lag 탐색 결과가 비어 있습니다. feature_cols 또는 target_col을 확인하세요.")

    # feature별 abs_corr가 가장 큰 lag 선택
    best_lag_df = (
        lag_result_df
        .sort_values(["feature", "abs_corr"], ascending=[True, False])
        .groupby("feature", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_lag_df = best_lag_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

    return lag_result_df, best_lag_df

def make_lagged_dataset_by_best_lag(
    df,
    best_lag_df,
    target_col,
    close_col,
    n_days=5,
    date_col="Date"
):
    """
    best_lag_df 기준으로 feature별 최적 lag를 적용한 최종 모델용 데이터셋 생성.
    """

    result = pd.DataFrame()
    result[date_col] = df[date_col]
    result[close_col] = df[close_col]

    # 확인용 미래수익률 보존
    future_ret_col = f"future_ret_{n_days}d"
    if future_ret_col in df.columns:
        result[future_ret_col] = df[future_ret_col]

    # target 보존
    result[target_col] = df[target_col]

    lagged_feature_cols = []

    for _, row in best_lag_df.iterrows():
        feature = row["feature"]
        lag = int(row["lag"])

        if feature not in df.columns:
            continue

        lagged_col = f"{feature}_lag{lag}"
        result[lagged_col] = df[feature].shift(lag)
        lagged_feature_cols.append(lagged_col)

    # 결측/무한값 제거
    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna().reset_index(drop=True)

    return result, lagged_feature_cols



################################################################################################################################################
## 5. RandomForest in-sample 학습 + permutation importance
################################################################################################################################################

def run_rf_permutation_importance_in_sample(
    lagged_df,
    feature_cols,
    target_col,
    date_col="Date",
    close_col=None,
    n_rf_runs=3,
    n_repeats=10,
    random_state=42
):
    """
    lagged_df 기준으로 RandomForestClassifier를 in-sample 학습한 뒤
    permutation importance를 반복 계산한다.

    핵심:
    - RF를 n_rf_runs번 학습
    - 각 RF마다 permutation을 n_repeats번 수행
    - feature별 importance raw 값을 전부 저장
    - 최종 importance_df는 총 n_rf_runs * n_repeats개의 raw importance 기준으로 계산

    예:
    n_rf_runs=3, n_repeats=10이면
    feature별 importance 값 30개를 기반으로 평균/표준편차/스코어 계산
    """

    df = lagged_df.copy()

    # =====================================================
    # 1. 모델 input / target 분리
    # =====================================================

    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X = X.replace([np.inf, -np.inf], np.nan)

    model_df = pd.concat([X, y], axis=1).dropna().copy()

    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int).copy()


    # =====================================================
    # 2. 여러 RF run + permutation raw importance 저장
    # =====================================================

    importance_records = []
    baseline_records = []

    for run in range(n_rf_runs):
        print()
        print(f"========== RF RUN {run + 1} / {n_rf_runs} ==========")

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state + run,
            n_jobs=-1
        )

        rf.fit(X, y)

        # =================================================
        # 3. in-sample 예측 성능 확인
        # =================================================

        pred = rf.predict(X)
        pred_proba = rf.predict_proba(X)[:, 1]

        acc = accuracy_score(y, pred)
        precision = precision_score(y, pred, zero_division=0)
        recall = recall_score(y, pred, zero_division=0)
        f1 = f1_score(y, pred, zero_division=0)
        loss = log_loss(y, pred_proba)

        cm = confusion_matrix(y, pred)

        print("accuracy :", round(acc, 4))
        print("precision:", round(precision, 4))
        print("recall   :", round(recall, 4))
        print("f1       :", round(f1, 4))
        print("log_loss :", round(loss, 4))

        baseline_records.append({
            "run": run + 1,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "log_loss": loss,
            "pred_1_count": int((pred == 1).sum()),
            "actual_1_count": int((y == 1).sum())
        })

        # =================================================
        # 4. permutation importance
        # =================================================

        perm = permutation_importance(
            rf,
            X,
            y,
            scoring="neg_log_loss",
            n_repeats=n_repeats,
            random_state=random_state + run,
            n_jobs=-1
        )

        # 핵심:
        # perm.importances shape = (n_features, n_repeats)
        # 여기서 반복별 raw importance를 전부 저장한다.
        for i, col in enumerate(feature_cols):
            for repeat_idx, importance_value in enumerate(perm.importances[i]):
                importance_records.append({
                    "run": run + 1,
                    "repeat": repeat_idx + 1,
                    "feature": col,
                    "importance": importance_value
                })

    # =====================================================
    # 5. 결과 정리
    # =====================================================

    raw_importance_df = pd.DataFrame(importance_records)
    baseline_df = pd.DataFrame(baseline_records)

    importance_df = (
        raw_importance_df
        .groupby("feature", as_index=False)
        .agg(
            importance_mean=("importance", "mean"),
            importance_std=("importance", "std"),
            importance_var=("importance", "var"),
            importance_min=("importance", "min"),
            importance_max=("importance", "max"),
            run_count=("run", "nunique"),
            repeat_count=("repeat", "count")
        )
    )

    # 안정성 점수
    # 평균 중요도는 높고, 30회 전체 기준 표준편차는 낮을수록 높게
    importance_df["importance_score"] = (
        importance_df["importance_mean"]
        - importance_df["importance_std"].fillna(0)
    )

    importance_df = importance_df.sort_values(
        "importance_score",
        ascending=False
    ).reset_index(drop=True)

    return importance_df, raw_importance_df, baseline_df

def split_lagged_feature_name(feature_name):
    """
    feature_lag20 형태의 컬럼명을 원본 feature와 lag로 분리한다.
    """

    if "_lag" not in feature_name:
        return feature_name, np.nan

    base_name = feature_name.rsplit("_lag", 1)[0]
    lag = feature_name.rsplit("_lag", 1)[1]

    try:
        lag = int(lag)
    except:
        lag = np.nan

    return base_name, lag



In [46]:
# =========================================================
# 1. target 없는 base dataset 생성
# =========================================================
print()
print("=" * 50)
print("1. Base feature dataset created.")

base_df, base_feature_cols, close_col = make_base_feature_dataset(
    etf_code=ETF_CODE,
    external_tickers=EXTERNAL_TICKERS,
    external_feature_types=EXTERNAL_FEATURE_TYPES,
    start_date=START_DATE,
    end_date=END_DATE
)

max_date = base_df["Date"].max()
valid_start_date = max_date - pd.DateOffset(months=VALID_MONTHS)

train_base_df = base_df[
    base_df["Date"] < valid_start_date
].copy()

valid_base_df = base_df[
    base_df["Date"] >= valid_start_date
].copy()

print("train_base_df shape:", train_base_df.shape)
print("valid_base_df shape:", valid_base_df.shape)
print("close_col:", close_col)
print("=" * 50)

# =========================================================
# 2. VIF 기반 불필요 칼럼 제거
# =========================================================
print()
print("=" * 50)
print("2. VIF filtering completed.")

vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
    df=train_base_df,
    feature_cols=base_feature_cols,
    vif_threshold=VIF_THRESHOLD,
    date_col="Date",
    verbose=True
)

# VIF 통과 feature만 남긴 데이터셋 생성
keep_cols = ["Date", close_col] + vif_feature_cols
vif_filtered_df = train_base_df[keep_cols].copy()

print("vif_filtered_df shape:", vif_filtered_df.shape)
print("제거된 컬럼:")
print(removed_vif_df)
print("=" * 50)

# =========================================================
# 3. Target 변수 생성
# =========================================================
print()
print("=" * 50)
print("3. Target column added.")

target_df, target_col = add_target_column(
    df=vif_filtered_df,
    close_col=close_col,
    n_days=N_DAYS,
    threshold=THRESHOLD
)

print("target_col:", target_col)

# target 없는 마지막 n_days 행 제거
target_df = target_df.dropna(subset=[target_col]).copy()
target_df[target_col] = target_df[target_col].astype(int)

print("target_df shape after dropna:", target_df.shape)
print("target 분포:")
print(target_df[target_col].value_counts())
print("target 비율:")
print(target_df[target_col].value_counts(normalize=True))

print("=" * 50)

# =========================================================
# 4.1. 최근 1년 데이터만 사용해서 lag 탐색
# =========================================================
print()
print("=" * 50)
print("4. 변수별 최적 LAG 탐색 완료.")

max_date = target_df["Date"].max()
lag_search_start_date = max_date - pd.DateOffset(years=LAG_SEARCH_YEARS)

target_df_for_lag_search = target_df[
    target_df["Date"] >= lag_search_start_date
].copy()

print("lag 탐색 기준 기간:")
print(target_df_for_lag_search["Date"].min(), "~", target_df_for_lag_search["Date"].max())
print("lag 탐색용 데이터 shape:", target_df_for_lag_search.shape)


# =========================================================
# 4.2. lag 탐색 실행
# =========================================================
exclude_cols_for_lag = [
    "Date",
    close_col,
    f"future_ret_{N_DAYS}d",
    target_col
]

lag_search_feature_cols = [
    col for col in target_df.columns
    if col not in exclude_cols_for_lag
]

lag_result_df, best_lag_df = find_best_lag_by_feature(
    df=target_df_for_lag_search,   # 핵심: 전체 target_df 말고 최근 1년만 넣음
    feature_cols=lag_search_feature_cols,
    target_col=target_col,
    lag_days=LAG_DAYS,
    date_col="Date"
)

print("전체 lag 탐색 결과 shape:", lag_result_df.shape)

# =========================================================
# 4.3. best lag 적용해서 lagged_df 생성
# =========================================================

lagged_df, lagged_feature_cols = make_lagged_dataset_by_best_lag(
    df=target_df,
    best_lag_df=best_lag_df,
    target_col=target_col,
    close_col=close_col,
    n_days=N_DAYS,
    date_col="Date"
)

print("lagged_df shape:", lagged_df.shape)
print("lagged feature count:", len(lagged_feature_cols))

print("=" * 50)


# =========================================================
# # 5.1. permutation importance 실행
# =========================================================
print()
print("=" * 50)
print("5. RandomForest in-sample 학습 + permutation importance 완료.")

importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
    lagged_df=lagged_df,
    feature_cols=lagged_feature_cols,
    target_col=target_col,
    date_col="Date",
    close_col=close_col,
    n_rf_runs=N_RF_RUNS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)


# =========================================================
# 5.2. 결과 feature명 / lag 분리해서 보기
# =========================================================

importance_view_df = importance_df.copy()

importance_view_df[["base_feature", "selected_lag"]] = importance_view_df["feature"].apply(
    lambda x: pd.Series(split_lagged_feature_name(x))
)

importance_view_df = importance_view_df[
    [
        "feature",
        "base_feature",
        "selected_lag",
        "importance_score",
        "importance_mean",
        "importance_std",
        "importance_var",
        "importance_min",
        "importance_max",
        "run_count",
        "repeat_count"
    ]
]

# =========================================================
# 5.3. best_lag_df와 importance 결과 합치기
# =========================================================

importance_with_lag_df = importance_view_df.merge(
    best_lag_df.rename(columns={
        "feature": "base_feature",
        "lag": "best_lag",
        "corr": "lag_corr",
        "abs_corr": "lag_abs_corr",
        "n_rows": "lag_n_rows"
    }),
    on="base_feature",
    how="left"
)

importance_with_lag_df = importance_with_lag_df[
    [
        "feature",
        "base_feature",
        "selected_lag",
        "importance_score",
        "importance_mean",
        "importance_std",
        "importance_var",
        "importance_min",
        "importance_max",
        "best_lag",
        "lag_corr",
        "lag_abs_corr",
        "lag_n_rows",
        "run_count",
        "repeat_count"
    ]
]

# 보기 좋게 정렬
importance_with_lag_df = importance_with_lag_df.sort_values(
    "importance_score",
    ascending=False
).reset_index(drop=True)

# =========================================================
# 5.4. TOP 변수 추출
# =========================================================

top_feature_df = importance_with_lag_df.head(TOP_N).copy()
top_feature_cols = top_feature_df["feature"].tolist()

print("TOP feature count:", len(top_feature_cols))
print(top_feature_cols)
print("=" * 50)

display(top_feature_df)



1. Base feature dataset created.
Loading external ticker: QQQ / QQQ
Loading external ticker: SPY / SPY
Loading external ticker: SOXX / SOXX
Loading external ticker: NVDA / NVDA
Loading external ticker: TSM / TSM
Loading external ticker: VIX / ^VIX
Loading external ticker: TNX / ^TNX
Loading external ticker: USDKRW / KRW=X
Loading external ticker: DXY / DX-Y.NYB
Loading external ticker: GOLD / GC=F
Loading external ticker: OIL / CL=F
train_base_df shape: (1579, 34)
valid_base_df shape: (23, 34)
close_col: SMH_adj_close

2. VIF filtering completed.
VIF 계산 대상 row 수: 1520
VIF 계산 대상 feature 수: 32
현재 max VIF: 156.10 / feature: SMH_ret_5d
현재 max VIF: 117.53 / feature: SMH_ret_20d
현재 max VIF: 15.11 / feature: QQQ_ret_20d
현재 max VIF: 12.11 / feature: QQQ_ret_5d
현재 max VIF: 10.15 / feature: SOXX_ret_20d
현재 max VIF: 9.32 / feature: SOXX_ret_5d

========== VIF 제거 결과 ==========
초기 feature 수: 32
상수 제거 feature 수: 0
VIF 제거 feature 수: 5
최종 feature 수: 27
vif_filtered_df shape: (1579, 29)
제거된 컬럼:
  remo

,feature,base_feature,selected_lag,importance_score,importance_mean,importance_std,importance_var,importance_min,importance_max,best_lag,lag_corr,lag_abs_corr,lag_n_rows,run_count,repeat_count
0,NVDA_ret_20d_lag1,NVDA_ret_20d,1,0.090840,0.094379,0.003539,1.252328e-05,0.087256,0.101184,1,-0.105986,0.105986,250,3,30
1,TNX_level_lag10,TNX_level,10,0.072540,0.074021,0.001481,2.193352e-06,0.070334,0.076658,10,0.245300,0.245300,241,3,30
2,TNX_diff_20d_lag10,TNX_diff_20d,10,0.069956,0.072280,0.002324,5.401601e-06,0.068603,0.076574,10,0.187152,0.187152,241,3,30
3,VIX_chg_5d_lag60,VIX_chg_5d,60,0.069777,0.071870,0.002093,4.379654e-06,0.067113,0.074630,60,0.206493,0.206493,191,3,30
4,SMH_ma20_ratio_lag10,SMH_ma20_ratio,10,0.067591,0.073172,0.005581,3.114535e-05,0.067315,0.082335,10,-0.094836,0.094836,241,3,30
5,USDKRW_ret_20d_lag1,USDKRW_ret_20d,1,0.065717,0.067815,0.002097,4.399476e-06,0.063579,0.071853,1,-0.201717,0.201717,250,3,30
6,GOLD_ret_5d_lag120,GOLD_ret_5d,120,0.065174,0.067024,0.001850,3.422069e-06,0.063702,0.070234,120,-0.261895,0.261895,131,3,30
7,OIL_ret_20d_lag60,OIL_ret_20d,60,0.056389,0.057931,0.001541,2.376122e-06,0.054610,0.061112,60,0.260864,0.260864,191,3,30
8,TNX_diff_5d_lag1,TNX_diff_5d,1,0.055789,0.057716,0.001928,3.716377e-06,0.052701,0.060666,1,-0.296878,0.296878,250,3,30
9,SPY_ret_20d_lag120,SPY_ret_20d,120,0.055695,0.057350,0.001655,2.739526e-06,0.054628,0.060362,120,-0.110836,0.110836,131,3,30


In [53]:
# =========================================================
# 6. valid_df 생성
# - base_df 전체에서 top_feature_df 기준 lag feature 생성
# - 마지막에 valid_base_df 길이만큼만 자름
# - 여기서는 dropna 절대 하지 않음
# - X, y 생성하지 않음
# =========================================================

print("=" * 60)
print("6. valid_df 생성")
print("=" * 60)

# ---------------------------------------------------------
# 1. top_feature_df 준비
# ---------------------------------------------------------

top_feature_df = top_feature_df.copy()

if ("base_feature" not in top_feature_df.columns) or ("selected_lag" not in top_feature_df.columns):
    top_feature_df[["base_feature", "selected_lag"]] = top_feature_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

top_feature_df["selected_lag"] = top_feature_df["selected_lag"].astype(int)

top_feature_cols = top_feature_df["feature"].tolist()
top_base_features = top_feature_df["base_feature"].unique().tolist()

print("top feature 수:", len(top_feature_cols))

# ---------------------------------------------------------
# 2. base_df에서 필요한 컬럼만 필터링
# ---------------------------------------------------------

need_cols = ["Date", close_col] + top_base_features

missing_cols = [col for col in need_cols if col not in base_df.columns]

if len(missing_cols) > 0:
    raise ValueError(f"base_df에 없는 컬럼이 있습니다: {missing_cols}")

valid_df = base_df[need_cols].copy()
valid_df = valid_df.sort_values("Date").reset_index(drop=True)

# ---------------------------------------------------------
# 3. 전체 데이터 기준으로 lag feature 생성
# ---------------------------------------------------------

for _, row in top_feature_df.iterrows():
    base_feature = row["base_feature"]
    selected_lag = int(row["selected_lag"])
    lagged_feature = row["feature"]

    valid_df[lagged_feature] = valid_df[base_feature].shift(selected_lag)

print("lag 생성 후 valid_df shape:", valid_df.shape)


# ---------------------------------------------------------
# 4. lag 생성 전 원본 base_feature 컬럼 제거
#    Date, close_col, top_feature_cols만 남김
# ---------------------------------------------------------

valid_df = valid_df[["Date", close_col] + top_feature_cols].copy()

print("원본 base_feature 제거 후 valid_df shape:", valid_df.shape)
print("최종 컬럼 수:", len(valid_df.columns))


# ---------------------------------------------------------
# 5. target 생성
# ---------------------------------------------------------

valid_df, _ = add_target_column(
    df=valid_df,
    close_col=close_col,
    n_days=N_DAYS,
    threshold=THRESHOLD,
    target_col=target_col
)

print("target 생성 후 valid_df shape:", valid_df.shape)


# ---------------------------------------------------------
# 6. 최근 valid_base_df 길이만큼 자르기
# ---------------------------------------------------------

valid_df = valid_df.tail(len(valid_base_df)).copy()
valid_df = valid_df.reset_index(drop=True)

print("최종 valid_df shape:", valid_df.shape)
print("valid_base_df shape:", valid_base_df.shape)

print("valid_df 기간:")
print(valid_df["Date"].min(), "~", valid_df["Date"].max())

# =========================================================
# 7. Train 학습 후 valid_df 예측
# =========================================================

print("=" * 60)
print("7. Train 학습 후 valid_df 예측")
print("=" * 60)

# ---------------------------------------------------------
# 1. top feature 컬럼 확인
# ---------------------------------------------------------

top_feature_cols = top_feature_df["feature"].tolist()

missing_train_cols = [col for col in top_feature_cols if col not in lagged_df.columns]
missing_valid_cols = [col for col in top_feature_cols if col not in valid_df.columns]

if len(missing_train_cols) > 0:
    raise ValueError(f"lagged_df에 없는 top feature가 있습니다: {missing_train_cols}")

if len(missing_valid_cols) > 0:
    raise ValueError(f"valid_df에 없는 top feature가 있습니다: {missing_valid_cols}")

print("사용 feature 수:", len(top_feature_cols))


# ---------------------------------------------------------
# 2. Train 데이터 준비
# ---------------------------------------------------------

train_df = lagged_df[
    ["Date", close_col, target_col] + top_feature_cols
].copy()

train_df = train_df.replace([np.inf, -np.inf], np.nan)
train_df = train_df.dropna(subset=top_feature_cols + [target_col]).copy()

X_train = train_df[top_feature_cols].copy()
y_train = train_df[target_col].astype(int).copy()

print("train_df shape:", train_df.shape)
print("X_train shape:", X_train.shape)

print("train target 분포:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))


# ---------------------------------------------------------
# 3. valid_df 예측용 데이터 준비
#    valid_df 행 수는 유지
#    단, 모델 입력용 X_valid_temp에는 결측 없는 행만 사용
# ---------------------------------------------------------

valid_pred_df = valid_df.copy()

valid_pred_df["pred_proba"] = np.nan
valid_pred_df["pred"] = np.nan

valid_available_mask = (
    valid_pred_df[top_feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .notna()
    .all(axis=1)
)

X_valid = valid_pred_df.loc[valid_available_mask, top_feature_cols].copy()

print("valid_df shape:", valid_df.shape)
print("예측 가능한 valid row 수:", len(X_valid))
print("예측 불가능 row 수:", len(valid_df) - len(X_valid))


# ---------------------------------------------------------
# 4. RandomForest 학습
# ---------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("모델 학습 완료")


# ---------------------------------------------------------
# 5. valid_df 예측
# ---------------------------------------------------------


valid_pred_proba = rf.predict_proba(X_valid)[:, 1]
valid_pred = (valid_pred_proba >= PRED_THRESHOLD).astype(int)

valid_pred_df.loc[valid_available_mask, "pred_proba"] = valid_pred_proba
valid_pred_df.loc[valid_available_mask, "pred"] = valid_pred

valid_pred_df["pred"] = valid_pred_df["pred"].astype("Int64")

print("valid_df 예측 완료")


# ---------------------------------------------------------
# 6. 예측 결과 확인
# ---------------------------------------------------------

display_cols = ["Date", close_col, "pred_proba", "pred"]

# target_col이 valid_df에 있으면 같이 보기
if target_col in valid_pred_df.columns:
    display_cols = ["Date", close_col, target_col, "pred_proba", "pred"]

display(valid_pred_df[display_cols])

print("예측값 분포:")
print(valid_pred_df["pred"].value_counts(dropna=False))

# =========================================================
# 7. 예측 1 기준 정확도 확인
# - pred = 1 인 것 중 실제 target = 1 인 비율
# - 우리가 주로 볼 precision 지표
# =========================================================

print("=" * 60)
print("7. 예측 1 기준 정확도 확인")
print("=" * 60)

# ---------------------------------------------------------
# 1. 평가 가능한 row만 사용
# - pred가 있어야 함
# - target도 있어야 함
# ---------------------------------------------------------

eval_df = valid_pred_df[
    valid_pred_df["pred"].notna() &
    valid_pred_df[target_col].notna()
].copy()

eval_df["pred"] = eval_df["pred"].astype(int)
eval_df[target_col] = eval_df[target_col].astype(int)

print("평가 가능 row 수:", len(eval_df))

# ---------------------------------------------------------
# 2. pred = 1인 row만 필터링
# ---------------------------------------------------------

pred_1_df = eval_df[eval_df["pred"] == 1].copy()

pred_1_count = len(pred_1_df)
pred_1_actual_1_count = (pred_1_df[target_col] == 1).sum()

print("예측 1 개수:", pred_1_count)
print("예측 1 중 실제 1 개수:", pred_1_actual_1_count)

# ---------------------------------------------------------
# 3. 예측 1 기준 정확도 = precision
# ---------------------------------------------------------

if pred_1_count > 0:
    pred_1_precision = pred_1_actual_1_count / pred_1_count
else:
    pred_1_precision = np.nan

print("예측 1 기준 정확도 precision:", pred_1_precision)

6. valid_df 생성
top feature 수: 27
lag 생성 후 valid_df shape: (1602, 56)
원본 base_feature 제거 후 valid_df shape: (1602, 29)
최종 컬럼 수: 29
target 생성 후 valid_df shape: (1602, 31)
최종 valid_df shape: (23, 31)
valid_base_df shape: (23, 34)
valid_df 기간:
2026-04-15 00:00:00 ~ 2026-05-15 00:00:00
7. Train 학습 후 valid_df 예측
사용 feature 수: 27
train_df shape: (1434, 30)
X_train shape: (1434, 27)
train target 분포:
target_5d_up_1pct
0    741
1    693
Name: count, dtype: int64
target_5d_up_1pct
0    0.516736
1    0.483264
Name: proportion, dtype: float64
valid_df shape: (23, 31)
예측 가능한 valid row 수: 23
예측 불가능 row 수: 0
모델 학습 완료
valid_df 예측 완료


,Date,SMH_adj_close,target_5d_up_1pct,pred_proba,pred
0,2026-04-15,453.000000,1.0,0.678,1
1,2026-04-16,454.799988,1.0,0.546,1
2,2026-04-17,464.160004,1.0,0.548,1
3,2026-04-20,463.959991,1.0,0.536,1
4,2026-04-21,464.660004,1.0,0.554,1
5,2026-04-22,476.829987,1.0,0.502,1
6,2026-04-23,481.850006,1.0,0.410,0
7,2026-04-24,506.440002,0.0,0.412,0
8,2026-04-27,506.260010,0.0,0.490,0
9,2026-04-28,491.209991,1.0,0.436,0


예측값 분포:
pred
1    15
0     8
Name: count, dtype: Int64
7. 예측 1 기준 정확도 확인
평가 가능 row 수: 18
예측 1 개수: 11
예측 1 중 실제 1 개수: 11
예측 1 기준 정확도 precision: 1.0
